# Deep Learning Training Notebook

This notebook trains a small neural network on the leaderboard CSV. It uses the same shared preprocessing helpers as the classical ML notebook.

In [ ]:
from pathlib import Path

import numpy as np

from tabular_utils import (
    TabularPreprocessor,
    extract_target,
    infer_target_column,
    load_csv_rows,
    regression_metrics,
    train_test_indices,
)

In [ ]:
CSV_PATH = Path("orbit-wars-publicleaderboard-2026-05-05T15:36:33.csv")
TARGET_COLUMN = None  # set to a specific column name if needed
DROP_COLUMNS = ["Rank", "TeamId"]
TEST_SIZE = 0.25
SEED = 42
HIDDEN_DIM = 16
EPOCHS = 1200
LR = 0.01
WEIGHT_DECAY = 1e-4

In [ ]:
class MLPRegressor:
    def __init__(self, input_dim: int, hidden_dim: int = 16, seed: int = 42) -> None:
        rng = np.random.default_rng(seed)
        self.w1 = rng.normal(0.0, 0.1, size=(input_dim, hidden_dim))
        self.b1 = np.zeros(hidden_dim)
        self.w2 = rng.normal(0.0, 0.1, size=(hidden_dim, 1))
        self.b2 = np.zeros(1)

    def forward(self, x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        z1 = x @ self.w1 + self.b1
        h1 = np.maximum(z1, 0.0)
        out = h1 @ self.w2 + self.b2
        return z1, h1, out

    def predict(self, x: np.ndarray) -> np.ndarray:
        _, _, out = self.forward(x)
        return out[:, 0]

    def fit(self, x: np.ndarray, y: np.ndarray, epochs: int = 1200, lr: float = 0.01, weight_decay: float = 1e-4) -> None:
        y_col = y.reshape(-1, 1)
        sample_count = x.shape[0]

        for epoch in range(epochs):
            z1, h1, out = self.forward(x)
            error = out - y_col
            loss = float(np.mean(error ** 2))

            grad_out = (2.0 / sample_count) * error
            grad_w2 = h1.T @ grad_out + weight_decay * self.w2
            grad_b2 = np.sum(grad_out, axis=0)

            grad_h1 = grad_out @ self.w2.T
            grad_z1 = grad_h1 * (z1 > 0.0)
            grad_w1 = x.T @ grad_z1 + weight_decay * self.w1
            grad_b1 = np.sum(grad_z1, axis=0)

            self.w2 -= lr * grad_w2
            self.b2 -= lr * grad_b2
            self.w1 -= lr * grad_w1
            self.b1 -= lr * grad_b1

            if epoch in {0, epochs - 1} or (epoch + 1) % max(1, epochs // 5) == 0:
                print(f"epoch={epoch + 1:4d} loss={loss:.6f}")

In [ ]:
columns, rows = load_csv_rows(str(CSV_PATH))
if not rows:
    raise ValueError(f"CSV file is empty: {CSV_PATH}")

target_column = TARGET_COLUMN or infer_target_column(columns)
if target_column not in columns:
    raise ValueError(f"Target column '{target_column}' not found in {CSV_PATH}.")

drop_columns = {column for column in DROP_COLUMNS if column in columns and column != target_column}
feature_columns = [column for column in columns if column != target_column and column not in drop_columns]
if not feature_columns:
    raise ValueError("No feature columns remain after dropping the target and excluded columns.")

train_idx, test_idx = train_test_indices(len(rows), test_size=TEST_SIZE, seed=SEED)
train_rows = [rows[index] for index in train_idx]
test_rows = [rows[index] for index in test_idx]

preprocessor = TabularPreprocessor()
x_train = preprocessor.fit_transform(train_rows, feature_columns)
x_test = preprocessor.transform(test_rows)
y_train = extract_target(train_rows, target_column)
y_test = extract_target(test_rows, target_column)

model = MLPRegressor(input_dim=x_train.shape[1], hidden_dim=HIDDEN_DIM, seed=SEED)
model.fit(x_train, y_train, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY)

predictions = model.predict(x_test)
metrics = regression_metrics(y_test, predictions)

print(f"CSV: {CSV_PATH}")
print(f"Target: {target_column}")
print(f"Features: {', '.join(feature_columns)}")
print(f"Train rows: {len(train_rows)} | Test rows: {len(test_rows)}")
print("Metrics:")
print(f"  MAE:  {metrics['mae']:.4f}")
print(f"  RMSE: {metrics['rmse']:.4f}")
print(f"  R2:   {metrics['r2']:.4f}")
print("Predictions vs actual:")
for actual, predicted in zip(y_test[:10], predictions[:10]):
    print(f"  actual={actual:.3f} predicted={predicted:.3f}")